كود التوليد و التدريب و الاختبار

In [1]:

import os
import random
import math
import json
from PIL import Image, ImageDraw
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from diffusers import StableDiffusionPipeline, DDPMScheduler
from accelerate import Accelerator
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model
from transformers import CLIPTokenizer

gc.collect()
torch.cuda.empty_cache()

# إعدادات عامة
shapes = ["circle", "ellipse", "square", "rectangle", "triangle", "line", "star",
          "rhombus", "parallelogram", "hexagon", "pentagon"]
IMAGE_SIZE = 128
SAMPLES_PER_SHAPE = 1000
BATCH_SIZE = 4  # قلل الحجم لتجنب نفاذ الذاكرة
EPOCHS = 5
PRETRAINED_MODEL = "./stable-diffusion-v1-4"
OUTPUT_DIR = "./trained_model"

# دوال توليد أشكال ... (تستطيع نسخ دوالك السابقة كما هي هنا)
def regular_polygon_vertices(center, radius, num_sides, rotation=0):
    cx, cy = center
    angle_step = 2 * math.pi / num_sides
    return [
        (
            cx + radius * math.cos(rotation + i * angle_step),
            cy + radius * math.sin(rotation + i * angle_step)
        )
        for i in range(num_sides)
    ]

def get_random_color():
    return tuple(random.randint(0, 255) for _ in range(3))

def get_random_position(width, height):
    margin = 10
    max_x = IMAGE_SIZE - width - margin
    max_y = IMAGE_SIZE - height - margin
    if max_x < margin or max_y < margin:
        # لا يمكن وضع الشكل ضمن الصورة بدون تخطي الحواف
        # نعيد مركز تقريبي بدلًا من الخطأ
        return margin, margin
    x = random.randint(margin, max_x)
    y = random.randint(margin, max_y)
    return x, y

def draw_shape(draw, shape, color, position, size):
    x, y = position
    center = (x + size // 2, y + size // 2)

    if shape == "circle":
        draw.ellipse([x, y, x+size, y+size], fill=color)
    elif shape == "square":
        draw.rectangle([x, y, x+size, y+size], fill=color)
    elif shape == "rectangle":
        draw.rectangle([x, y, x+size*2, y+size], fill=color)
    elif shape == "ellipse":
        draw.ellipse([x, y, x+size*2, y+size], fill=color)
    elif shape == "triangle":
        draw.polygon([(x, y+size), (x+size//2, y), (x+size, y+size)], fill=color)
    elif shape == "line":
        draw.line([x, y, x+size, y+size], fill=color, width=3)
    elif shape == "star":
        points = regular_polygon_vertices(center, size//2, 5, rotation=math.pi/2)
        draw.polygon(points, fill=color)
    elif shape == "rhombus":
        draw.polygon([
            (x+size//2, y),
            (x+size, y+size//2),
            (x+size//2, y+size),
            (x, y+size//2)
        ], fill=color)
    elif shape == "parallelogram":
        draw.polygon([
            (x+size//3, y),
            (x+size, y),
            (x+2*size//3, y+size),
            (x, y+size)
        ], fill=color)
    elif shape == "hexagon":
        points = regular_polygon_vertices(center, size//2, 6, rotation=math.pi/6)
        draw.polygon(points, fill=color)
    elif shape == "pentagon":
        points = regular_polygon_vertices(center, size//2, 5, rotation=math.pi/2)
        draw.polygon(points, fill=color)

def generate_and_save_data():
    os.makedirs("generated_images", exist_ok=True)
    metadata = []
    print("🚀 بدء توليد البيانات وحفظها على القرص ...")

    for shape in shapes:
        for i in tqdm(range(SAMPLES_PER_SHAPE), desc=f"Generating {shape}"):
            bg_color = get_random_color()
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), bg_color)
            draw = ImageDraw.Draw(img)
            size = random.randint(20, 60)
            color = get_random_color()

            if shape in ["rectangle", "ellipse"]:
                width = size * 2
                height = size
            elif shape == "parallelogram":
                width = size
                height = size
            else:
                width = height = size

            position = get_random_position(width, height)
            draw_shape(draw, shape, color, position, size)

            description = f"A {shape} with color RGB{color}, size {size} at position {position}, on background RGB{bg_color}."
            filename = f"{shape}_{i}.png"
            img.save(f"generated_images/{filename}")
            metadata.append({"filename": filename, "description": description})

    with open("metadata.json", "w") as f:
        json.dump(metadata, f)
    print(f"✅ تم توليد وحفظ {len(metadata)} صور مع الأوصاف")


# تعريف Dataset
class ShapeDataset(Dataset):
    def __init__(self, metadata_file, images_folder, tokenizer, augment=False):
        with open(metadata_file, "r") as f:
            self.metadata = json.load(f)
        self.images_folder = images_folder
        self.tokenizer = tokenizer
        self.augment = augment
        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(30),
                transforms.ColorJitter(0.3, 0.3, 0.3),
                transforms.ToTensor(),
                transforms.Normalize([0.5] * 3, [0.5] * 3)
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize([0.5] * 3, [0.5] * 3)
            ])

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        img_path = os.path.join(self.images_folder, item["filename"])
        description = item["description"]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        tokenized = self.tokenizer(description, padding="max_length", truncation=True,
                                   max_length=77, return_tensors="pt")
        return {
            "pixel_values": img,
            "input_ids": tokenized.input_ids.squeeze(0),
            "attention_mask": tokenized.attention_mask.squeeze(0)
        }

def evaluate(model, dataloader, device):
    model.unet.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in dataloader:
            images = batch["pixel_values"].to(device, dtype=model.unet.dtype)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            latents = model.vae.encode(images).latent_dist.sample() * 0.18215
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, model.scheduler.config.num_train_timesteps,
                                      (latents.shape[0],), device=device).long()

            noisy_latents = model.scheduler.add_noise(latents, noise, timesteps)
            encoder_hidden_states = model.text_encoder(input_ids, attention_mask=attention_mask).last_hidden_state
            noise_pred = model.unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            total_loss += loss.item()
    model.unet.train()
    return total_loss / len(dataloader)

def train():
    if not os.path.exists("metadata.json") or len(os.listdir("generated_images")) < len(shapes)*SAMPLES_PER_SHAPE:
        generate_and_save_data()

    if not os.path.exists("metadata.json"):
        print("يجب توليد البيانات أولاً")
        return

    with open("metadata.json") as f:
        metadata = json.load(f)

    train_meta, test_meta = train_test_split(metadata, test_size=0.2, random_state=42)
    json.dump(train_meta, open("train_metadata.json", "w"))
    json.dump(test_meta, open("test_metadata.json", "w"))
    print(f"بيانات التدريب: {len(train_meta)}، بيانات الاختبار: {len(test_meta)}")

    # تحميل نموذج Stable Diffusion الأساسي
    pipe = StableDiffusionPipeline.from_pretrained(
        PRETRAINED_MODEL,
        torch_dtype=torch.float16,
        revision="fp16"
    )
    tokenizer = pipe.tokenizer

    # تكوين LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_q",
                        "down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_k",
                        "down_blocks.0.attentions.1.transformer_blocks.0.attn1.to_v"],  # الطبقات التي ستُطبق عليها LoRA في UNet
        lora_dropout=0.05,
        bias="none",
        # لا تضف task_type هنا لأنه يسبب الخطأ في هذا السياق
    )

    pipe.unet = get_peft_model(pipe.unet, lora_config)
    pipe.unet.print_trainable_parameters()

    train_dataset = ShapeDataset("train_metadata.json", "generated_images", tokenizer, augment=True)
    test_dataset = ShapeDataset("test_metadata.json", "generated_images", tokenizer, augment=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    accelerator = Accelerator(mixed_precision="fp16")
    pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

    optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=5e-6)

    # تجهيز جميع الكائنات للتدريب مع Accelerator
    train_loader, test_loader, optimizer, pipe.unet = accelerator.prepare(train_loader, test_loader, optimizer, pipe.unet)
    device = accelerator.device
    pipe.to(device)
    print(f"using device:{device}")
    print("بدء التدريب ...")
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        loop = tqdm(train_loader, leave=False)
        for batch in loop:
            images = batch["pixel_values"].to(device, dtype=pipe.unet.dtype)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            with accelerator.accumulate(pipe.unet):
                latents = pipe.vae.encode(images).latent_dist.sample() * 0.18215
                noise = torch.randn_like(latents)
                timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
                noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
                encoder_hidden_states = pipe.text_encoder(input_ids, attention_mask=attention_mask).last_hidden_state
                noise_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample
                
                if torch.isnan(latents.detach().float().cpu()).any(): print("NaN latents")
                if torch.isnan(noise.detach().float().cpu()).any(): print("NaN noise")
                if torch.isnan(noisy_latents.detach().float().cpu()).any(): print("NaN noisy_latents")
                if torch.isnan(encoder_hidden_states.detach().float().cpu()).any(): print("NaN encoder_hidden_states")
                if torch.isnan(noise_pred.detach().float().cpu()).any(): print("NaN noise_pred")
                loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float())
            
                #loss = torch.nn.functional.mse_loss(noise_pred, noise)
                if torch.isnan(loss) or torch.isinf(loss):
                    print("Batch تخطى بسبب NaN أو Inf في الخسارة")
                    optimizer.zero_grad()
                    continue

                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad()

                loop.set_postfix(loss=loss.item())

        test_loss = evaluate(pipe, test_loader, device)
        print(f"خسارة التقييم: {test_loss:.4f}")
        gc.collect()
        torch.cuda.empty_cache()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
# حفظ LoRA فقط (الوحدات القابلة للتدريب)
    pipe.unet.save_pretrained(OUTPUT_DIR)

# اختياري: حفظ الـ tokenizer والـ scheduler و text_encoder إذا أردت تشغيل النموذج مباشرة لاحقًا
    pipe.tokenizer.save_pretrained(OUTPUT_DIR)
    pipe.text_encoder.save_pretrained(OUTPUT_DIR)
    pipe.scheduler.save_pretrained(OUTPUT_DIR)
    pipe.vae.save_pretrained(OUTPUT_DIR)
    pipe.safety_checker.save_pretrained(OUTPUT_DIR)
    pipe.feature_extractor.save_pretrained(OUTPUT_DIR)
    print(f"تم حفظ النموذج في: {OUTPUT_DIR}")

if __name__ == "__main__":
    train()


C:\Users\ComputerWorld\anaconda3\envs\ai_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


بيانات التدريب: 8800، بيانات الاختبار: 2200


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

trainable params: 15,360 || all params: 859,536,324 || trainable%: 0.0018
using device:cuda
بدء التدريب ...
Epoch 1/5


خسارة التقييم: 0.1597
Epoch 2/5


خسارة التقييم: 0.1124
Epoch 3/5


خسارة التقييم: 0.0595
Epoch 4/5


خسارة التقييم: 0.0494
Epoch 5/5


خسارة التقييم: 0.0459
تم حفظ النموذج في: ./trained_model


اظهار الطبقات وتحديدها

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe=StableDiffusionPipeline.from_pretrained(
    "./stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    revision="fp16"
)
pipe.to("cpu")

print("\n الطبقات القابلة لتطبيق lora (linear/ conv2d)\n")
for name, module in pipe.unet.named_modules():
    if isinstance(module,(torch.nn.Linear,torch.nn.Conv2d)):
        print(f"{name} -> {type(module)}")

كود دمج 

In [2]:

from diffusers import StableDiffusionPipeline
from peft import PeftModel
import torch

# تحميل النموذج الأساسي (الذي استخدمته في التدريب)
base_model_path = "./stable-diffusion-v1-4"  # أو "./stable-diffusion-v1-4" إذا كان محليًا
lora_model_path = "./trained_model"  # هذا هو مجلد LoRA الناتج من التدريب

pipe = StableDiffusionPipeline.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16
)

# تحميل LoRA وتطبيقه
pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_model_path)

# دمج LoRA داخل UNet وإلغاء الاعتماد على Peft
pipe.unet = pipe.unet.merge_and_unload()

# حفظ النموذج النهائي الكامل
merged_path = "./merged_model"
pipe.save_pretrained(merged_path)

print(f"✅ تم دمج النموذج وتخزينه في: {merged_path}")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

✅ تم دمج النموذج وتخزينه في: ./merged_model


كود واجهة Gradio

In [12]:
import torch
from diffusers import StableDiffusionPipeline
import gradio as gr

# إذا النموذج محلي
MODEL_PATH = "./merged_model"

# أو لتحميل من Hugging Face 
# MODEL_PATH = "username/model_name"

# تحميل النموذج
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,
   # safety_checker=None
).to(device)

def generate_image(prompt, num_inference_steps=50, guidance_scale=7.5):
    with torch.autocast(device):
        image = pipe(prompt, num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).images[0]
    return image

# واجهة Gradio
iface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="أدخل الوصف النصي", lines=2, placeholder="مثلاً: a red circle on a blue background"),
        gr.Slider(minimum=10, maximum=100, step=1, value=30, label="عدد خطوات التوليد (Inference steps)"),
        gr.Slider(minimum=1.0, maximum=20.0, step=0.5, value=7, label="مقياس التوجيه (Guidance scale)")
    ],
    outputs=gr.Image(type="pil"),
    title="مولد صور الأشكال الهندسية",
    description="أدخل وصفًا نصيًا لشكل هندسي وسيتم توليد صورة باستخدام نموذج Stable Diffusion مدرب."
)

iface.launch()

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


واجهة API

In [13]:
# ملف: main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image
import uuid
import os

# تحميل النموذج من مجلد محلي (تأكد من تعديل المسار حسب مكان النموذج المدرب)
model_path = "./merged_model"
pipe = StableDiffusionPipeline.from_pretrained(model_path, torch_dtype=torch.float16).to("cuda" if torch.cuda.is_available() else "cpu")
pipe.safety_checker = None  # لإزالة فلتر الصور غير المرغوبة إذا كنت متأكدًا من البيانات

# مجلد حفظ الصور الناتجة
os.makedirs("generated_images", exist_ok=True)

# إنشاء تطبيق FastAPI
app = FastAPI()

# نموذج البيانات المستلمة
class TextPrompt(BaseModel):
    prompt: str

@app.post("/generate")
def generate_image(request: TextPrompt):
    try:
        # توليد الصورة
        image: Image.Image = pipe(request.prompt).images[0]

        # حفظ الصورة باسم فريد
        filename = f"{uuid.uuid4().hex}.png"
        save_path = os.path.join("generated_images", filename)
        image.save(save_path)

        return {"message": "✅ Image generated successfully", "filename": filename}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]